# Análise de características usando uma Árvore de Decisão

> **Autores:** Alexandre Maciel e Vinicius de Lima.

Esse notebook calcula o F1-score de classificações feitas usando uma Árvore de Decisão nas bases descritas na seguinte tabela:

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier

In [ ]:
sorted = pd.read_csv("knn/sorted.csv")
sorted

In [ ]:
par_criterion = ["gini", "entropy", "log_loss"]
par_max_depth = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]


def executar_experimentos(X, y):
    resultados_cv = []

    for c in par_criterion:
        for d in par_max_depth:
            clf = DecisionTreeClassifier(criterion=c, max_depth=d, random_state=42)
            scores = cross_val_score(clf, X, y, cv=10, scoring="f1_macro")
            resultados_cv.append(
                {"criterion": c, "max_depth": d, "f1_cv": np.mean(scores)}
            )

    df_resultados = pd.DataFrame(resultados_cv)
    top_10 = df_resultados.sort_values(by="f1_cv", ascending=False).head(10)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=42
    )

    resultados_finais = []
    for index, row in top_10.iterrows():
        clf = DecisionTreeClassifier(
            criterion=row["criterion"], max_depth=int(row["max_depth"]), random_state=42
        )
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        f1_holdout = f1_score(y_test, y_pred, average="macro")

        resultados_finais.append(
            {
                "criterion": row["criterion"],
                "max_depth": int(row["max_depth"]),
                "f1_10fold": row["f1_cv"],
                "f1_70_30": f1_holdout,
            }
        )

    return pd.DataFrame(resultados_finais)


configs = []
resultados = []
for i, dataset in enumerate(sorted["dataset"]):
    df = pd.read_csv(f"bases/{dataset}")
    X = df.drop(columns=["label"])
    y = df["label"]

    print(f"executando experimentos para dataset {dataset}")
    print(f"{i + 1} de {len(sorted['dataset'])}")

    experimento = executar_experimentos(X, y)

    conf = {"dataset": dataset}
    resultado_10_fold = {"dataset": dataset, "split": "10-fold"}
    resultado_70_30 = {"dataset": dataset, "split": "70/30"}
    for i, r in experimento.iterrows():
        idx = f"conf. {i + 1}"
        conf[idx] = f"criterion={r['criterion']}, max_depth={r['max_depth']}"
        resultado_10_fold[idx] = r["f1_10fold"]
        resultado_70_30[idx] = r["f1_70_30"]

    resultados.extend([resultado_10_fold, resultado_70_30])
    configs.append(conf)

configs = pd.DataFrame(configs)
resultados = pd.DataFrame(resultados)

In [9]:
display(configs)
display(resultados)
resultados.describe()

,dataset,conf. 1,conf. 2,conf. 3,conf. 4,conf. 5,conf. 6,conf. 7,conf. 8,conf. 9,conf. 10
0,PCA_075_HOG_256_32x32.csv,"criterion=gini, max_depth=10","criterion=gini, max_depth=8","criterion=log_loss, max_depth=9","criterion=entropy, max_depth=9","criterion=entropy, max_depth=6","criterion=log_loss, max_depth=6","criterion=gini, max_depth=4","criterion=entropy, max_depth=8","criterion=log_loss, max_depth=8","criterion=log_loss, max_depth=7"
1,PCA_075_HOG_256_32x32.csv,"criterion=gini, max_depth=10","criterion=gini, max_depth=8","criterion=log_loss, max_depth=9","criterion=entropy, max_depth=9","criterion=entropy, max_depth=6","criterion=log_loss, max_depth=6","criterion=gini, max_depth=4","criterion=entropy, max_depth=8","criterion=log_loss, max_depth=8","criterion=log_loss, max_depth=7"
2,PCA_075_HOG_128_16x16.csv,"criterion=gini, max_depth=3","criterion=gini, max_depth=2","criterion=log_loss, max_depth=3","criterion=entropy, max_depth=3","criterion=gini, max_depth=4","criterion=gini, max_depth=5","criterion=gini, max_depth=14","criterion=entropy, max_depth=2","criterion=log_loss, max_depth=2","criterion=gini, max_depth=6"
3,PCA_090_HOG_256_32x32.csv,"criterion=gini, max_depth=4","criterion=entropy, max_depth=2","criterion=log_loss, max_depth=2","criterion=gini, max_depth=5","criterion=entropy, max_depth=8","criterion=log_loss, max_depth=8","criterion=gini, max_depth=6","criterion=entropy, max_depth=3","criterion=log_loss, max_depth=3","criterion=gini, max_depth=2"
4,PCA_090_HOG_256_32x32.csv,"criterion=gini, max_depth=4","criterion=entropy, max_depth=2","criterion=log_loss, max_depth=2","criterion=gini, max_depth=5","criterion=entropy, max_depth=8","criterion=log_loss, max_depth=8","criterion=gini, max_depth=6","criterion=entropy, max_depth=3","criterion=log_loss, max_depth=3","criterion=gini, max_depth=2"
5,HOG_256_32x32.csv,"criterion=gini, max_depth=10","criterion=gini, max_depth=12","criterion=gini, max_depth=9","criterion=gini, max_depth=2","criterion=gini, max_depth=5","criterion=gini, max_depth=14","criterion=gini, max_depth=13","criterion=gini, max_depth=15","criterion=gini, max_depth=11","criterion=gini, max_depth=6"
6,PCA_090_HOG_128_16x16.csv,"criterion=gini, max_depth=3","criterion=entropy, max_depth=3","criterion=log_loss, max_depth=3","criterion=gini, max_depth=2","criterion=entropy, max_depth=2","criterion=log_loss, max_depth=2","criterion=gini, max_depth=5","criterion=entropy, max_depth=4","criterion=log_loss, max_depth=4","criterion=gini, max_depth=6"
7,PCA_075_HOG_128_32x32.csv,"criterion=log_loss, max_depth=4","criterion=entropy, max_depth=4","criterion=gini, max_depth=14","criterion=gini, max_depth=13","criterion=gini, max_depth=12","criterion=entropy, max_depth=7","criterion=log_loss, max_depth=7","criterion=gini, max_depth=11","criterion=gini, max_depth=15","criterion=gini, max_depth=9"
8,HOG_256_32x32.csv,"criterion=gini, max_depth=10","criterion=gini, max_depth=12","criterion=gini, max_depth=9","criterion=gini, max_depth=2","criterion=gini, max_depth=5","criterion=gini, max_depth=14","criterion=gini, max_depth=13","criterion=gini, max_depth=15","criterion=gini, max_depth=11","criterion=gini, max_depth=6"
9,PCA_075_HOG_128_16x16.csv,"criterion=gini, max_depth=3","criterion=gini, max_depth=2","criterion=log_loss, max_depth=3","criterion=entropy, max_depth=3","criterion=gini, max_depth=4","criterion=gini, max_depth=5","criterion=gini, max_depth=14","criterion=entropy, max_depth=2","criterion=log_loss, max_depth=2","criterion=gini, max_depth=6"


,dataset,split,conf. 1,conf. 2,conf. 3,conf. 4,conf. 5,conf. 6,conf. 7,conf. 8,conf. 9,conf. 10
0,PCA_075_HOG_256_32x32.csv,10-fold,0.610033,0.606656,0.606210,0.606210,0.605023,0.605023,0.602752,0.602354,0.602354,0.601916
1,PCA_075_HOG_256_32x32.csv,70/30,0.575920,0.603333,0.633333,0.633333,0.624583,0.624583,0.606996,0.574882,0.574882,0.595489
2,PCA_075_HOG_256_32x32.csv,10-fold,0.610033,0.606656,0.606210,0.606210,0.605023,0.605023,0.602752,0.602354,0.602354,0.601916
3,PCA_075_HOG_256_32x32.csv,70/30,0.575920,0.603333,0.633333,0.633333,0.624583,0.624583,0.606996,0.574882,0.574882,0.595489
4,PCA_075_HOG_128_16x16.csv,10-fold,0.664426,0.640556,0.638249,0.638249,0.637134,0.635714,0.632561,0.630013,0.630013,0.628732
5,PCA_075_HOG_128_16x16.csv,70/30,0.641948,0.624896,0.649734,0.649734,0.617908,0.624060,0.587149,0.624896,0.624896,0.603609
6,PCA_090_HOG_256_32x32.csv,10-fold,0.596060,0.595886,0.595886,0.594508,0.592648,0.592648,0.591725,0.589350,0.589350,0.588002
7,PCA_090_HOG_256_32x32.csv,70/30,0.591638,0.640443,0.640443,0.586919,0.591212,0.591212,0.595826,0.645532,0.645532,0.619346
8,PCA_090_HOG_256_32x32.csv,10-fold,0.596060,0.595886,0.595886,0.594508,0.592648,0.592648,0.591725,0.589350,0.589350,0.588002
9,PCA_090_HOG_256_32x32.csv,70/30,0.591638,0.640443,0.640443,0.586919,0.591212,0.591212,0.595826,0.645532,0.645532,0.619346


,conf. 1,conf. 2,conf. 3,conf. 4,conf. 5,conf. 6,conf. 7,conf. 8,conf. 9,conf. 10
count,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000
mean,0.613534,0.618198,0.621045,0.616680,0.615726,0.612109,0.606071,0.612734,0.609660,0.610737
std,0.027426,0.022150,0.023307,0.024420,0.019488,0.019322,0.020174,0.024084,0.021970,0.017000
min,0.555245,0.555245,0.548464,0.548464,0.582289,0.582289,0.575266,0.570707,0.570707,0.585185
25%,0.595871,0.605825,0.606966,0.603290,0.602185,0.592648,0.591725,0.595067,0.597150,0.599624
50%,0.610033,0.620827,0.628644,0.619904,0.616402,0.609560,0.602752,0.612840,0.607573,0.607960
75%,0.626646,0.640443,0.638249,0.633333,0.632992,0.624661,0.614051,0.631871,0.628793,0.628473
max,0.664426,0.649611,0.649734,0.649734,0.649976,0.645827,0.653439,0.645827,0.645532,0.636990


# 